# Credit Risk Assessment: Predicting Credit Card Default

**IDRA Data Science & AI Capstone Project**

This notebook reproduces the analysis used in the final report. It covers data loading, quality checks, cleaning, feature engineering, EDA, statistical summaries, classification modelling, evaluation, and feature importance.

**Target:** `default.payment.next.month`  
**Models:** Logistic Regression, Decision Tree, Random Forest  
**Split:** 80/20 stratified train-test split, `random_state=42`


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

df = pd.read_csv("P_5_UCI_Credit_Card.csv")
df.head()


In [ ]:
print("Shape:", df.shape)
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
display(df.describe().T)
display(df["default.payment.next.month"].value_counts())


## Data Cleaning and Feature Engineering

The supplied data contains no missing values or duplicate rows. Category codes are harmonised for modelling: education codes 0, 5 and 6 are grouped into the documented 'other' category, and marriage code 0 is grouped into 'other'. Additional aggregate variables summarise historical bills, payments, utilisation and repayment delay.

In [ ]:
target = "default.payment.next.month"
data = df.copy()

data["EDUCATION_CLEAN"] = data["EDUCATION"].replace({0: 4, 5: 4, 6: 4})
data["MARRIAGE_CLEAN"] = data["MARRIAGE"].replace({0: 3})

bill_cols = [f"BILL_AMT{i}" for i in range(1, 7)]
pay_cols = [f"PAY_AMT{i}" for i in range(1, 7)]
paystat_cols = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]

data["AVG_BILL_AMT"] = data[bill_cols].mean(axis=1)
data["AVG_PAY_AMT"] = data[pay_cols].mean(axis=1)
data["TOTAL_BILL_AMT"] = data[bill_cols].sum(axis=1)
data["TOTAL_PAY_AMT"] = data[pay_cols].sum(axis=1)
data["UTILIZATION_1"] = np.where(
    data["LIMIT_BAL"] > 0, data["BILL_AMT1"] / data["LIMIT_BAL"], 0
)
data["MAX_PAY_DELAY"] = data[paystat_cols].max(axis=1)
data["AVG_PAY_DELAY"] = data[paystat_cols].mean(axis=1)

data.head()


## Exploratory Data Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
data[target].value_counts().sort_index().plot(kind="bar", ax=ax)
ax.set_title("Target Class Distribution")
ax.set_xlabel("Default in next month")
ax.set_ylabel("Number of accounts")
plt.tight_layout()
plt.show()

pay0_summary = data.groupby("PAY_0")[target].agg(["count", "mean"])
display(pay0_summary)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(pay0_summary.index, pay0_summary["mean"] * 100, marker="o")
ax.set_title("Observed Default Rate by PAY_0")
ax.set_xlabel("PAY_0")
ax.set_ylabel("Default rate (%)")
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
corr_cols = [
    "LIMIT_BAL", "AGE", "PAY_0", "PAY_2", "PAY_3",
    "BILL_AMT1", "BILL_AMT2", "BILL_AMT3",
    "PAY_AMT1", "PAY_AMT2", "PAY_AMT3", target
]
plt.figure(figsize=(8.5, 7))
sns.heatmap(data[corr_cols].corr(), cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()


## Statistical Analysis

In [ ]:
key_nums = [
    "LIMIT_BAL", "AGE", "AVG_BILL_AMT", "AVG_PAY_AMT",
    "UTILIZATION_1", "MAX_PAY_DELAY", "TOTAL_PAY_AMT"
]
display(data.groupby(target)[key_nums].agg(["mean", "median"]).round(2))


## Model Development

In [ ]:
X = data.drop(columns=["ID", "EDUCATION", "MARRIAGE", target])
y = data[target]

cat_features = ["SEX", "EDUCATION_CLEAN", "MARRIAGE_CLEAN"]
num_features = [c for c in X.columns if c not in cat_features]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

models = {
    "Logistic Regression": Pipeline([
        ("prep", preprocessor),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced"))
    ]),
    "Decision Tree": Pipeline([
        ("prep", preprocessor),
        ("model", DecisionTreeClassifier(
            max_depth=6, min_samples_leaf=20,
            class_weight="balanced", random_state=42
        ))
    ]),
    "Random Forest": Pipeline([
        ("prep", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=300, max_depth=10, min_samples_leaf=5,
            class_weight="balanced", random_state=42, n_jobs=-1
        ))
    ])
}


In [ ]:
results = {}
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]

    results[name] = {
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1-score": f1_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, proba)
    }
    predictions[name] = (pred, proba)

results_df = pd.DataFrame(results).T
display(results_df.round(4))


In [ ]:
best_model = models["Random Forest"]
best_pred = predictions["Random Forest"][0]

cm = confusion_matrix(y_test, best_pred)
sns.heatmap(cm, annot=True, fmt="d", cbar=False)
plt.title("Random Forest Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

feature_names = best_model.named_steps["prep"].get_feature_names_out()
importances = best_model.named_steps["model"].feature_importances_
feature_importance = pd.Series(importances, index=feature_names).sort_values(ascending=False)
display(feature_importance.head(15))


## Reproducibility Notes

- The dataset used in this notebook is the supplied `P_5_UCI_Credit_Card.csv`.
- The original UCI repository describes the dataset as a 30,000-instance, 23-feature classification dataset with no missing values.
- All preprocessing transformations used by the model are fitted through the training pipeline to reduce leakage risk.
- The report should be read together with this notebook because the notebook is the detailed computational record.
